# LAG và rollback

Dữ liệu thật BTS tháng01/2025.

**Câu hỏi:** Ngày hôm trước có ít chuyến hơn không?

**Dự đoán trước:** mỗi dòng kết quả là một chuyến hay một nhóm?

In [ ]:
from pathlib import Path
import sys
root=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'flight_lab').is_dir())
sys.path.insert(0,str(root))
from flight_lab.learning import database,read_sql
db=database()
print('Database chỉ đọc:',db)


Tổng hợp ngày trước rồi mới LAG; nguồn có đủ31ngày trong snapshot này.

In [ ]:
read_sql('WITH daily AS(SELECT flight_date,COUNT(*) total FROM flights GROUP BY flight_date) SELECT flight_date,total,LAG(total) OVER(ORDER BY flight_date) previous_total FROM daily ORDER BY flight_date')

Thực hành ghi bằng database memory GIẢ LẬP; không ghi nguồn thật.

In [ ]:
import sqlite3
with sqlite3.connect(':memory:') as practice:
    practice.execute('CREATE TABLE demo(id INTEGER PRIMARY KEY,n INTEGER CHECK(n>0))')
    try:
        practice.execute('BEGIN IMMEDIATE')
        practice.execute('INSERT INTO demo VALUES(1,1)')
        practice.execute('INSERT INTO demo VALUES(2,-1)')
        practice.commit()
    except sqlite3.IntegrityError:practice.rollback()
    assert practice.execute('SELECT COUNT(*) FROM demo').fetchone()[0]==0
print('Không lưu dòng đầu nếu thao tác sau lỗi')

## Tự làm

Giải thích ngày đầu LAG làNULL. Thử COMMIT với hai dòng hợp lệ trong bảng tạm.

Xem lời giải riêng sau khi thử. Không ghi vào database thật.